# Solution E3 · KFold on data sorted by class and `error_score=0` hiding everything

**Type**: Find the bug

## Objective

Understand why a cross-validation returns an AUC of 0 on all the folds without raising errors, and fix the splitter.
The colleague's code has two defects that hide each other: data sorted by target and `error_score=0`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False).sort_values("defaulted").reset_index(drop=True)      # rows sorted by target
X_sorted = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y_sorted = df["defaulted"].to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Commented solution

In [ ]:
bug_scores = cross_val_score(model, X_sorted, y_sorted, cv=KFold(5), scoring="roc_auc", error_score=0)
auc_bug = bug_scores.mean()
print("AUC per fold:", bug_scores.round(3), "| mean:", round(auc_bug, 3))

**Where the bug is.** With the rows sorted by target, `KFold(5)` (without shuffling) creates five contiguous blocks: in the first four validation
folds there is a single class (the AUC is not defined) and in the training set of the fifth almost all the examples are of the same class (the
model cannot even be trained). Every fold raises an error, and `error_score=0` silently turns it into the score 0.

In [ ]:
try:
    cross_val_score(model, X_sorted, y_sorted, cv=KFold(5), scoring="roc_auc", error_score="raise")
except ValueError as error:
    message = str(error).splitlines()[0]
print("real error:", message)

bug_positives_per_fold = [int(y_sorted[va].sum()) for _, va in KFold(5).split(X_sorted)]
print("positives in the validation folds (KFold without shuffle):", bug_positives_per_fold)

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=0)
auc_ok = cross_val_score(model, X_sorted, y_sorted, cv=skf, scoring="roc_auc").mean()
ok_positives_per_fold = [int(y_sorted[va].sum()) for _, va in skf.split(X_sorted, y_sorted)]
print(f"correct AUC: {auc_ok:.3f} | positives per fold: {ok_positives_per_fold}")

## Check

In [ ]:
with verify():
    check(auc_bug == 0.0, "the colleague's number is 0.0 on all the folds: they are errors turned into scores", "don't modify the colleague's code")
    check(isinstance(message, str) and ("class" in message.lower() or "one" in message.lower()),
          "the error message talks about a single class", "with error_score='raise' cross_val_score raises a ValueError about the classes")
    check(bug_positives_per_fold is not None and sorted(bug_positives_per_fold)[:4] == [0, 0, 0, 0] and max(bug_positives_per_fold) > 200,
          "four validation folds have no positives at all, the last one almost only positives", "with KFold without shuffle the folds are contiguous blocks of sorted rows")
    check(0.7 < auc_ok < 0.85, "with stratification the AUC is the true one (between 0.7 and 0.85)", "StratifiedKFold(5, shuffle=True, random_state=0) without error_score")
    check(ok_positives_per_fold is not None and max(ok_positives_per_fold) - min(ok_positives_per_fold) <= 1,
          "with stratification the positives are evenly distributed (difference at most 1)", "every fold must have about 19% positives")
done()

## What you learned

- `KFold` without `shuffle` divides into contiguous blocks: with rows sorted by class, by date or by group the folds are not
  representative. For classification you use `StratifiedKFold(shuffle=True)`, which is also the default of `cross_val_score` when you pass
  an integer (but not if you build `KFold` by hand).
- `error_score=0` (or `np.nan`) turns errors into numbers. It is useful in a search where some combinations may fail, and
  very dangerous elsewhere: an error you don't see is an error you don't fix. Use `"raise"` while you are developing.
- A score of exactly 0.0 (or exactly 0.5) on all the folds is almost always a mistake, not a performance.

## Common pitfall

Sorting or grouping the data "for convenience" (by class, by customer, by date) before using `KFold` without shuffling. Cross-validation
assumes the rows are interchangeable: if they are not, the splitter must be chosen deliberately (groups, time, stratification).

## Interview question

<details><summary>When is it legitimate to use `error_score=np.nan` in a hyperparameter search?</summary>

When some combinations can legitimately fail (for example a solver that does not converge or an incompatible combination of parameters)
and I don't want to interrupt the whole search. The failed combinations get NaN and are excluded from the ranking. But I always check how many
failed (`cv_results_`) and investigate: if they all fail, or one particular fold fails, the error has a cause (data, splitter) that I am hiding.

</details>